# 01. ETH yield와 BTC–ETH perpetual funding spread

이 노트북은 **ETH staking/LST-implied yield가 BTC 대비 ETH 무기한선물 funding cost에 가격 반영되는지**를 실험한다.

핵심 spread는 사용자가 제안한 정의에 맞춰 다음처럼 둔다.

```text
S_t = f_BTC,t - f_ETH,t
```

따라서 `S_t`가 하락하면 ETH funding이 BTC funding보다 상대적으로 높아진 것이고, `S_t`가 상승하면 ETH funding이 상대적으로 낮아진 것이다.

두 경쟁 채널을 모두 열어둔다.

| 계수 부호 | 가능한 우세 채널 |
| --- | --- |
| `β < 0` | dividend-like carry channel: ETH yield가 spot/LST 보유 매력을 높이고 ETH funding을 BTC 대비 상승 |
| `β > 0` | delta-neutral basis-trade channel: LST/spot long + ETH perp short 수요가 커져 ETH funding 하락 |
| `β ≈ 0` | 직접 가격 반영이 약하거나 두 채널이 상쇄 |


In [ ]:
import sys
from pathlib import Path


def resolve_project_root(start: Path) -> Path:
    """Find project root containing `common` and `data` directories."""
    for candidate in [start, *start.parents]:
        if (candidate / "common").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError("Could not resolve project root from current working directory")


PROJECT_ROOT = resolve_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"PROJECT_ROOT={PROJECT_ROOT}")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from common.transforms import funding_to_daily_annualized, log_return
from common.visualization import plot_timeseries


## 1. 실험 설정

`POST_START`는 regime-dependent formulation에서 사용할 post-period 시작일이다. 연구 설계에 맞춰 Merge, Shanghai, ETF 승인일, 표본 중간값 등으로 바꿔 실험할 수 있다.

`YIELD_WINDOW_DAYS`는 `share_rate`/`exchange_rate`가 있을 때 log realized yield를 계산하는 창이다.


In [ ]:
RAW = PROJECT_ROOT / "data" / "raw"
PROCESSED = PROJECT_ROOT / "data" / "processed"
RESULTS = PROJECT_ROOT / "data" / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

POST_START = "2024-01-11"  # 예: spot BTC ETF approval 이후. 필요시 변경.
YIELD_WINDOW_DAYS = 7       # wstETH share/exchange-rate realized yield window k.
HAC_MAXLAGS = 7             # daily regression HAC lag.


## 2. 데이터 로드

필요 파일:

- ETH/BTC funding CSV: `data/raw/binance_funding_*.csv` 또는 `bybit_funding_*.csv`
- ETH/BTC 1d price CSV: `data/raw/binance_klines_*_1d.csv` 또는 legacy 파일명
- ETH yield CSV: `data/processed/eth_yield_panel.csv`, `data/processed/lido_wsteth_share_rate.csv`, 또는 `data/raw/eth_staking_yield_daily.csv`


In [ ]:
def first_existing(paths):
    return next((p for p in paths if p.exists()), None)


eth_f_path = first_existing([
    RAW / "binance_funding_ETHUSDT.csv",
    RAW / "bybit_funding_ETHUSDT.csv",
    RAW / "binance_ethusdt_funding.csv",
    RAW / "bybit_ethusdt_funding.csv",
])
btc_f_path = first_existing([
    RAW / "binance_funding_BTCUSDT.csv",
    RAW / "bybit_funding_BTCUSDT.csv",
    RAW / "binance_btcusdt_funding.csv",
    RAW / "bybit_btcusdt_funding.csv",
])
eth_p_path = first_existing([
    RAW / "binance_klines_ETHUSDT_1d.csv",
    RAW / "binance_ethusdt_1d.csv",
])
btc_p_path = first_existing([
    RAW / "binance_klines_BTCUSDT_1d.csv",
    RAW / "binance_btcusdt_1d.csv",
])
st_path = first_existing([
    PROCESSED / "eth_yield_panel.csv",
    PROCESSED / "lido_wsteth_share_rate.csv",
    RAW / "eth_staking_yield_daily.csv",
    RAW / "cf_eth_srr_assumed.example.csv",  # smoke-test fallback only
])

required_paths = {
    "ETH funding": eth_f_path,
    "BTC funding": btc_f_path,
    "ETH price": eth_p_path,
    "BTC price": btc_p_path,
    "ETH yield": st_path,
}
missing = [name for name, path in required_paths.items() if path is None]
if missing:
    raise FileNotFoundError(f"Missing required data files: {missing}. Run the data collection scripts first.")

eth_f = pd.read_csv(eth_f_path)
btc_f = pd.read_csv(btc_f_path)
eth_p = pd.read_csv(eth_p_path)
btc_p = pd.read_csv(btc_p_path)
st = pd.read_csv(st_path)

required_paths


## 3. 분석 패널 구성

핵심 변수:

```text
f_BTC,t = BTC perpetual funding rate
f_ETH,t = ETH perpetual funding rate
S_t     = f_BTC,t - f_ETH,t
r_t     = Δ log(P_ETH,t / P_BTC,t)
RV_t    = rolling realized volatility of r_t
```

`share_rate` 또는 `exchange_rate`가 yield 패널에 있으면 다음 proxy도 만든다.

```text
y_ETH,t(k) = (365 / k) × [log(R_t) - log(R_{t-k})]
```


In [ ]:
def normalize_daily_date(series: pd.Series) -> pd.Series:
    """Normalize mixed date/timestamp inputs to naive daily datetime."""
    if pd.api.types.is_numeric_dtype(series):
        numeric = pd.to_numeric(series, errors="coerce")
        unit = "ms" if numeric.dropna().abs().median() > 1e11 else "s"
        parsed = pd.to_datetime(numeric, unit=unit, utc=True, errors="coerce")
    else:
        parsed = pd.to_datetime(series, utc=True, errors="coerce", format="mixed")
        if parsed.isna().any():
            numeric = pd.to_numeric(series, errors="coerce")
            if numeric.notna().any():
                unit = "ms" if numeric.dropna().abs().median() > 1e11 else "s"
                parsed = parsed.fillna(pd.to_datetime(numeric, unit=unit, utc=True, errors="coerce"))
    return parsed.dt.tz_convert(None).dt.normalize()


def standardize_price_frame(df: pd.DataFrame, label: str) -> pd.DataFrame:
    """Return daily close frame with columns date and close."""
    out = df.copy()
    date_col = next((c for c in ["date", "open_time", "openTime", "timestamp", "time"] if c in out.columns), None)
    close_col = next((c for c in ["close", "Close", "price", "last"] if c in out.columns), None)
    if date_col is None or close_col is None:
        raise KeyError(f"{label} price file needs a date-like column and a close-like column. columns={out.columns.tolist()}")
    out["date"] = normalize_daily_date(out[date_col])
    out["close"] = pd.to_numeric(out[close_col], errors="coerce")
    return out[["date", "close"]].dropna().sort_values("date")


def standardize_yield_frame(df: pd.DataFrame, k: int = 7) -> pd.DataFrame:
    """Return daily ETH yield frame with stake_yield plus optional realized share-rate yield."""
    out = df.copy()
    date_col = next((c for c in ["date", "Date", "day", "timestamp", "time"] if c in out.columns), None)
    if date_col is None:
        raise KeyError(f"Yield file needs a date-like column. columns={out.columns.tolist()}")
    out["date"] = normalize_daily_date(out[date_col])

    yield_col = next((c for c in ["stake_yield", "annualized_apr_decimal", "eth_srr", "reward_rate", "value"] if c in out.columns), None)
    if yield_col is None:
        raise KeyError(f"Yield file needs one of stake_yield/annualized_apr_decimal/eth_srr/reward_rate/value. columns={out.columns.tolist()}")
    out["stake_yield"] = pd.to_numeric(out[yield_col], errors="coerce")
    if out["stake_yield"].dropna().median() > 1:
        out["stake_yield"] = out["stake_yield"] / 100.0

    rate_col = next((c for c in ["share_rate", "exchange_rate", "wsteth_exchange_rate", "steth_per_token"] if c in out.columns), None)
    if rate_col is not None:
        out[rate_col] = pd.to_numeric(out[rate_col], errors="coerce")
        out[f"stake_yield_log_{k}d"] = (365.0 / k) * (np.log(out[rate_col]) - np.log(out[rate_col]).shift(k))
    else:
        out[f"stake_yield_log_{k}d"] = np.nan

    return out[["date", "stake_yield", f"stake_yield_log_{k}d"]].dropna(subset=["date"]).sort_values("date")


eth_fd = funding_to_daily_annualized(eth_f, time_col="fundingTime", rate_col="fundingRate")
btc_fd = funding_to_daily_annualized(btc_f, time_col="fundingTime", rate_col="fundingRate")

funding = eth_fd[["date", "funding_ann"]].merge(
    btc_fd[["date", "funding_ann"]], on="date", how="inner", suffixes=("_eth", "_btc")
).rename(columns={"funding_ann_eth": "f_eth", "funding_ann_btc": "f_btc"})
funding["spread_btc_minus_eth"] = funding["f_btc"] - funding["f_eth"]

eth_px = standardize_price_frame(eth_p, "ETH")
btc_px = standardize_price_frame(btc_p, "BTC")
prices = eth_px.merge(btc_px, on="date", how="inner", suffixes=("_eth", "_btc"))
prices["ret_eth_btc"] = log_return(prices["close_eth"] / prices["close_btc"])
prices["rv_eth_btc"] = prices["ret_eth_btc"].rolling(7, min_periods=2).std() * np.sqrt(365.0)

yield_panel = standardize_yield_frame(st, k=YIELD_WINDOW_DAYS)

df = (
    funding.merge(prices[["date", "ret_eth_btc", "rv_eth_btc"]], on="date", how="inner")
    .merge(yield_panel, on="date", how="inner")
    .dropna(subset=["spread_btc_minus_eth", "f_eth", "f_btc", "stake_yield", "ret_eth_btc", "rv_eth_btc"])
    .sort_values("date")
    .reset_index(drop=True)
)

df["post"] = (df["date"] >= pd.Timestamp(POST_START)).astype(int)
df["stake_yield_x_post"] = df["stake_yield"] * df["post"]

df.head(), df.describe(include="all")


In [ ]:
# 기초 시계열 시각화
fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
plot_timeseries(df, "date", "spread_btc_minus_eth", "BTC - ETH Funding Spread (annualized)", ax=axes[0])
plot_timeseries(df, "date", "f_eth", "ETH Funding (annualized)", ax=axes[1])
plot_timeseries(df, "date", "f_btc", "BTC Funding (annualized)", ax=axes[2])
plot_timeseries(df, "date", "stake_yield", "ETH Yield Proxy (annualized)", ax=axes[3])
plt.tight_layout()


In [ ]:
# 분석 변수 상관관계
corr_cols = ["spread_btc_minus_eth", "f_eth", "f_btc", "stake_yield", "ret_eth_btc", "rv_eth_btc"]
if df[f"stake_yield_log_{YIELD_WINDOW_DAYS}d"].notna().any():
    corr_cols.append(f"stake_yield_log_{YIELD_WINDOW_DAYS}d")
df[corr_cols].corr()


## 4. 공통 회귀 함수

모든 모델은 daily data의 serial correlation 가능성을 고려해 HAC/Newey-West 표준오차를 사용한다.


In [ ]:
def fit_hac(data: pd.DataFrame, y_col: str, x_cols: list[str], maxlags: int = HAC_MAXLAGS):
    model_df = data[[y_col, *x_cols]].replace([np.inf, -np.inf], np.nan).dropna()
    y = model_df[y_col]
    X = sm.add_constant(model_df[x_cols], has_constant="add")
    res = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": maxlags})
    return res, model_df


def tidy_result(res, model_name: str, y_col: str) -> pd.DataFrame:
    out = pd.DataFrame({
        "model": model_name,
        "dependent": y_col,
        "term": res.params.index,
        "coef": res.params.values,
        "std_err_hac": res.bse.values,
        "t": res.tvalues.values,
        "p_value": res.pvalues.values,
        "nobs": int(res.nobs),
        "adj_r2": res.rsquared_adj,
    })
    return out


def slope_interpretation(beta: float, p: float, alpha: float = 0.10) -> str:
    if p >= alpha:
        return "유의하지 않음: yield pricing evidence가 약하거나 상쇄 가능"
    if beta < 0:
        return "β < 0: dividend-like carry channel 우세 가능"
    return "β > 0: delta-neutral basis-trade/hedge channel 우세 가능"


## 5. Model A — 단순 baseline

```text
S_t = α + β y_ETH,t + ε_t
```

- `β < 0`: ETH yield 상승 시 ETH funding이 BTC 대비 높아져 `BTC - ETH` spread 하락
- `β > 0`: LST/spot long + ETH perp short 수요가 우세해 ETH funding이 낮아져 spread 상승


In [ ]:
res_a, sample_a = fit_hac(df, "spread_btc_minus_eth", ["stake_yield"])
print(res_a.summary())
print(slope_interpretation(res_a.params["stake_yield"], res_a.pvalues["stake_yield"]))


## 6. Model B — controls 포함 baseline

```text
S_t = α + β y_ETH,t + γ₁ r_ETH/BTC,t + γ₂ RV_ETH/BTC,t + ε_t
```

상대가격 momentum과 변동성이 funding spread를 직접 움직일 수 있으므로 통제한다.


In [ ]:
controls = ["ret_eth_btc", "rv_eth_btc"]
res_b, sample_b = fit_hac(df, "spread_btc_minus_eth", ["stake_yield", *controls])
print(res_b.summary())
print(slope_interpretation(res_b.params["stake_yield"], res_b.pvalues["stake_yield"]))


## 7. Model C — funding leg decomposition

Spread만 보면 BTC leg가 만든 착시일 수 있으므로 ETH funding과 BTC funding을 각각 추정한다.

```text
f_ETH,t = α + β_ETH y_ETH,t + controls + ε_t
f_BTC,t = α + β_BTC y_ETH,t + controls + ε_t
S_t     = f_BTC,t - f_ETH,t
```

해석 기준:

| 결과 | 해석 |
| --- | --- |
| `β_ETH > 0`, `β_BTC ≈ 0`, spread `β < 0` | ETH yield가 ETH funding에 반영되는 가장 직접적인 패턴 |
| `β_ETH ≈ 0`, spread `β` 유의 | BTC leg 움직임이 spread를 만든 가능성 |
| 둘 다 유의 | 공통 시장요인 또는 omitted variable 점검 필요 |
| 둘 다 비유의 | yield pricing evidence 약함 |


In [ ]:
res_c_eth, sample_c_eth = fit_hac(df, "f_eth", ["stake_yield", *controls])
res_c_btc, sample_c_btc = fit_hac(df, "f_btc", ["stake_yield", *controls])

print("ETH funding leg")
print(res_c_eth.summary())
print("\nBTC funding leg")
print(res_c_btc.summary())

leg_summary = pd.concat([
    tidy_result(res_c_eth, "C1_ETH_leg", "f_eth"),
    tidy_result(res_c_btc, "C2_BTC_leg", "f_btc"),
])
leg_summary.query("term == 'stake_yield'")


## 8. Model D — regime-dependent formulation

```text
S_t = α + β₁ y_ETH,t + β₂ Post_t + β₃ y_ETH,t × Post_t + γ controls_t + ε_t
```

- Pre-period slope: `β₁`
- Post-period slope: `β₁ + β₃`

`POST_START` 값을 바꿔서 시기별로 관계가 달라지는지 반복 실험한다.


In [ ]:
regime_x = ["stake_yield", "post", "stake_yield_x_post", *controls]
res_d, sample_d = fit_hac(df, "spread_btc_minus_eth", regime_x)
print(res_d.summary())

pre_slope = res_d.params["stake_yield"]
post_slope = res_d.params["stake_yield"] + res_d.params["stake_yield_x_post"]
print(f"Pre-period slope β₁: {pre_slope:.6f}")
print(f"Post-period slope β₁+β₃: {post_slope:.6f}")


## 9. Model E — wstETH share/exchange-rate realized yield proxy

Yield panel에 `share_rate` 또는 `exchange_rate`가 있을 때 protocol exchange/share rate에서 계산한 realized yield를 사용한다.

```text
y_ETH,t(k) = (365 / k) × [log(R_t) - log(R_{t-k})]
```

해당 열이 없으면 이 모델은 자동으로 skip된다.


In [ ]:
realized_yield_col = f"stake_yield_log_{YIELD_WINDOW_DAYS}d"
if df[realized_yield_col].notna().sum() < 20:
    print(f"Skip Model E: {realized_yield_col} has too few non-missing observations.")
    res_e = None
else:
    res_e, sample_e = fit_hac(df, "spread_btc_minus_eth", [realized_yield_col, *controls])
    print(res_e.summary())
    print(slope_interpretation(res_e.params[realized_yield_col], res_e.pvalues[realized_yield_col]))


## 10. 모델 결과 요약 및 저장

`data/results/01_funding_vs_staking_model_results.csv`에 모든 회귀 계수표를 저장한다.


In [ ]:
result_tables = [
    tidy_result(res_a, "A_simple_spread", "spread_btc_minus_eth"),
    tidy_result(res_b, "B_controls_spread", "spread_btc_minus_eth"),
    tidy_result(res_c_eth, "C1_ETH_leg", "f_eth"),
    tidy_result(res_c_btc, "C2_BTC_leg", "f_btc"),
    tidy_result(res_d, "D_regime_spread", "spread_btc_minus_eth"),
]
if res_e is not None:
    result_tables.append(tidy_result(res_e, "E_realized_yield_spread", "spread_btc_minus_eth"))

results = pd.concat(result_tables, ignore_index=True)
out_path = RESULTS / "01_funding_vs_staking_model_results.csv"
results.to_csv(out_path, index=False)

key_terms = ["stake_yield", "stake_yield_x_post", f"stake_yield_log_{YIELD_WINDOW_DAYS}d"]
print(f"Saved: {out_path}")
results[results["term"].isin(key_terms)].sort_values(["model", "term"])


## 11. 최종 해석 템플릿

보고서/논문에는 아래처럼 정리한다.

> ETH yield may affect the BTC–ETH funding spread through two competing channels. Under the dividend-like carry channel, higher ETH yield raises the relative attractiveness of ETH spot/LST exposure and should increase ETH funding relative to BTC, leading to a negative coefficient in a regression of BTC-minus-ETH funding spread on ETH yield. Under the delta-neutral basis-trade channel, higher ETH yield may instead increase demand for spot/LST-long and perp-short strategies, putting downward pressure on ETH funding and producing a positive coefficient. The empirical analysis therefore tests whether ETH yield is priced in the BTC–ETH funding spread, and uses the sign of the coefficient to distinguish the dominant channel.

한국어:

> ETH yield는 두 가지 상반된 경로를 통해 BTC–ETH funding spread에 영향을 줄 수 있다. 첫째, 배당 유사 carry 경로에서는 ETH yield가 ETH spot/LST 보유의 상대적 매력을 높여 ETH funding을 BTC 대비 상승시키므로, `BTC funding - ETH funding`에 대한 회귀계수는 음수가 예상된다. 둘째, delta-neutral basis trade 경로에서는 ETH yield 상승이 spot/LST long 및 perp short 전략의 매력을 높여 ETH funding에 하방 압력을 가할 수 있으므로, 회귀계수는 양수가 될 수 있다. 따라서 본 연구는 ETH yield가 BTC–ETH funding spread에 가격 반영되는지를 검증하고, 계수의 부호를 통해 어떤 경로가 우세한지 해석한다.
